In [ ]:
import os
import requests
from dotenv import load_dotenv, find_dotenv
_=load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]
weatherstack_api_key = os.environ["WEATHERSTACK_API_KEY"]

print(f"openai_api_key: {openai_api_key}")
print(f"weatherstack_api_key: {weatherstack_api_key}")

In [ ]:

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from pydantic import BaseModel

In [ ]:
search_tool = TavilySearch(max_result=2)

class CurrentWeather(BaseModel):
    temperature: float
    weather_description: list[str]
    humidity: int

class WeatherResponse(BaseModel):
    current: CurrentWeather

@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city."""

    response = requests.get(
        "http://api.weatherstack.com/current",
        params={"access_key": weatherstack_api_key, "query": city},
        timeout=10,
    )

    response.raise_for_status()

    data = response.json()
    
    weather = CurrentWeather.model_validate(data)

    return (
        f"City: {city}\n",
        f"Temperature: {weather.current['temperature']}°C\n"
        f"Weather: {weather.current['weather_descriptions'][0]}\n"
        f"Humidity: {weather.current['humidity']}%",
    )


llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

tools = [search_tool, get_weather_data]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. "
        "Use the available tools whenever external or "
        "current information is required."
    ),
)

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Find the capital of India"
                "and then find its current weather."
            )
        }
    ]
})

print("\n========================")
print("FINAL OUTPUT")
print("========================\n")

print(response["messages"][-1].content)